In [9]:
#进入Nature主页，通过局级检索功能，搜索关键词llm, 限制年份为“2023-2024"，搜索得到近两年间关键词含有llm的Nature平台相关文章。
# Q.使用python代码获取检索结果第一页的html内容并解码为文本串，将其以UTF-8编码格式写入page1.txt文件中，用于后续处理。

import requests

url = 'https://www.nature.com/search?q=llm&order=relevance&date_range=2023-2024'
response = requests.get(url)
#获取page1的html

if response.status_code == 200:
    data = response.text
else:
    print('Request failed with status code:',response.status_code)
#解码为字符串data

with open('page1.txt','w', encoding = 'UTF_8') as file1:
    file1.write(data)
#写入文件page1.txt

In [10]:
#本页面展示了搜索结果中前几条内容的截图
# Q.打开page1.txt, 观察相关数据的组织格式规律。从这些文本串中提取一些论文相关的重要信息(包括文章标题，文章地址，文章简介，作者列表，文章类型，期刊名称，卷宗/页面信息) 
# 并按照发表的期刊名进行分类，以下面的格式存储到一个字典列表中。
# Q.基于上述结果，输出每个期刊在pagel中包含的论文数量

import re

paperlist = []

with open('page1.txt','r', encoding = 'UTF_8') as file1:
    start = 0
    #标记article块是否开始
    end = 0
    #标记article块是否结束
    authors = []
    description = ''
    for line in file1:
        if not start:
            start = re.search('<article',line)
            #匹配article块开头
        else:
            end = re.search('</article>',line)
            #匹配article块结尾
            if end:
                if description == '' :
                    description = 'no description'
                #处理没有descreption的情况
                flag = 0
                #标记期刊是否存在
                for dict1 in paperlist:
                    if dict1["journal"] == journal:
                       (dict1["papers"]).append({"title": title, "authors": authors, "url": paperurl, "description": description, "type": type, "volume_page_info": page}) 
                       flag = 1
                       break
                if flag == 0:
                    paperlist.append({"journal": journal, "papers": [{"title": title, "authors": authors, "url": paperurl, "description": description, "type": type, "volume_page_info": page}]})
                start = 0
                end = 0
                authors = []
                description = ''
                #找到结尾，存储所有信息到paperlist，重置start,end,authors和description
            else:
                matcht = re.search('data-track-label="link">(.*)</a>',line)
                if matcht:
                    title = matcht.group(1)
                #匹配论文标题
                matchdes = re.search('<p>(.*)</p>',line)
                if matchdes:
                    description = matchdes.group(1)
                #匹配论文描述
                matchurl = re.search('<a href="(.*)"',line)
                if matchurl:
                    paperurl = matchurl.group(1)
                #匹配论文url
                authors += re.findall('<span itemprop="name">(.*?)</span>',line)
                #匹配论文作者
                matchtype = re.search('<span class="c-meta__type">(.*?)</span>',line)
                if matchtype:
                    type = matchtype.group(1)
                #匹配论文类型
                matchjournal = re.search('data-test="journal-title-and-link">(.*?)</div>',line)
                if matchjournal:
                    journal = matchjournal.group(1)
                #匹配论文类型
                matchpage = re.search('data-test="volume-and-page-info">(.*?)</div>',line)
                if matchpage:
                    page = matchpage.group(1)
                #匹配论文类型
#完成信息存储

for dict1 in paperlist:
    print(f'期刊名：{dict1['journal']}，论文数量：{len(dict1['papers'])}')
#输出每个期刊在page1中包含的论文数量

期刊名：Nature Machine Intelligence，论文数量：6
期刊名：Nature Communications，论文数量：10
期刊名：Scientific Reports，论文数量：12
期刊名：Nature Methods，论文数量：1
期刊名：npj Digital Medicine，论文数量：3
期刊名：Nature Medicine，论文数量：2
期刊名：Nature，论文数量：4
期刊名：Nature Human Behaviour，论文数量：2
期刊名：npj Precision Oncology，论文数量：1
期刊名：BDJ Open，论文数量：1
期刊名：Nature Computational Science，论文数量：2
期刊名：Nature Reviews Urology，论文数量：1
期刊名：Communications Materials，论文数量：1
期刊名：npj Biodiversity，论文数量：1
期刊名：Humanities and Social Sciences Communications，论文数量：1
期刊名：Eye，论文数量：1
期刊名：npj Computational Materials，论文数量：1


In [11]:
#下面展示的是某篇文章打开后的界面
# Q. 观察可以发现该文章的作者信息实际上多于搜索结果中展示的内容，请你仔细观察此界面的html数据组织格式，依据此编写python程序，
# 将上一步骤中的字典提取内容中的作者列表中的内容进行替换，替换为文章主页面显示的全部作者

journallist = []
# 存储期刊代码与期数，用于任务5

for dict1 in paperlist:
    for paper1 in dict1["papers"]:
        urlp = paper1["url"]
        url = 'https://www.nature.com' + urlp
        #取出上述论文信息中的url后缀，加上前缀得到详情页的url
        response = requests.get(url)
        if response.status_code == 200:
            data = response.text
        else:
            print('Request failed with status code:',response.status_code)
        #得到论文详情页的html代码

        match = re.search('"authors":(.*?),"publishedAt"',data)
        #匹配作者
        if match:
            authors = eval(match.group(1))
            #作者在html编码格式中为list形式，直接读入列表表达式并用eval函数转换即可
            paper1["authors"] = authors
            #更新作者信息
    
        matchjournal = re.search('"journal":{"pcode":"(.*?)","title":"([Nn]ature(?! R[r]eviews).*?)","volume"',data)
        matchvolume = re.search('"volume":"(.*?)","issue"',data)
        #为任务5作准备，匹配期刊代码、期数与期刊名，期刊名以Nature/nature开头且不以Nature Reviews开头
        #实际上Nature Reviews开头的期刊html内会因为volume格式为"volume":null而不被匹配到
        if matchjournal and matchvolume:
            journal = matchjournal.group(1)
            volume = matchvolume.group(1)
            journalname = matchjournal.group(2)
            if (journal, volume, journalname) not in journallist:
                journallist.append((journal, volume, journalname))
#更新完成

In [12]:
#JSON是一种轻量级的数据交互格式，常用于存储和表示数据。
# Q.将上一步获得的字典列表转化为 json 对象（可使用json包的相应方法），并以2字符缩进的方式写入nature_llm.json文件中

import json

with open('nature_llm.json','w',encoding = 'UTF_8') as fj:
    json.dump(paperlist, fj, indent=2)
    #以2字符缩进将列表存储到nature_llm.json文件中

In [13]:
#（选做）Nature平台的Advanced Reasearch页面中，还可以依据期刊名称和Volume进行检索
#Q.对于上述提取到的pagel中的文章，请你尝试提取所有发表在以Nature为开头的期刊（不包括Nature Reviews开头的期刊）的文章
# 请你检索得到这些文章所在的相应Volume中的所有文章信息，提取检索结果的pagel中的所有文章，并以如下格式存储在volume.json文件中
# 重复的Volume只需要提取一次。查找过程需要使用python代码自动实现，不可以手动获取网址。 
#（此处的author信息只使用检索界面的部分信息即可）

volumelist = []

for journal, volume, journalname in journallist:
    newdict = {"journal": journalname, "volume": volume, "papers": []}

    url = f'https://www.nature.com/search?volume={volume}&order=relevance&journal={journal}'
    response = requests.get(url)
    if response.status_code == 200:
        data = response.text
    else:
        print('Request failed with status code:',response.status_code)
    #获取网页html

    pattern = re.compile(r'<article\s+class[^>]*>(.*?)</article>', re.DOTALL)
    matches = re.findall(pattern,data)
    #找到所有article块

    authors = []
    description = ''
    for article in matches:
        matcht = re.search('data-track-label="link">(.*)</a>',article)
        if matcht:
            title = matcht.group(1)
        #匹配论文标题
        matchdes = re.search('<p>(.*)</p>',article)
        if matchdes:
            description = matchdes.group(1)
        #匹配论文描述
        matchurl = re.search('<a href="(.*)"',article)
        if matchurl:
            paperurl = matchurl.group(1)
        #匹配论文url
        authors += re.findall('<span itemprop="name">(.*?)</span>',article)
        #匹配论文作者
        matchtype = re.search('<span class="c-meta__type">(.*?)</span>',article)
        if matchtype:
            type = matchtype.group(1)
        #匹配论文类型
        
        if description == '' :
            description = 'no description'
        #处理没有descreption的情况

        (newdict["papers"]).append({"title": title, "authors": authors, "url": paperurl, "description": description, "type": type})
        #更新newdict内papers的内容

        authors = []
        description = ''
        #重置authors和description

    volumelist.append(newdict)

with open('volume.json','w',encoding = 'UTF_8') as fj:
    json.dump(volumelist, fj, indent=2)
    #以2字符缩进将列表存储到volume.json文件中
#完成信息存储
